In [0]:
# Nesta fase, vamos fazer transformações nos dados e limpeza. Análise exploratória para garantir qualidade dos dados, eliminando problemas clássicos da engenharia (como null, números com vírgula, etc)

display (spark.sql("""
    SELECT COUNT(*) AS linhas,
    SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS partida_real_null_de_verdade,
    SUM(CASE WHEN partida_real = 'null' THEN 1 ELSE 0 END) AS partida_real_string_null,
    SUM(CASE WHEN partida_prevista = 'null' THEN 1 ELSE 0 END) AS partida_prevista_string_null,
    SUM(CASE WHEN partida_prevista LIKE '%.%' THEN 1 ELSE 0 END) AS com_fracao_de_segundo
    FROM voebem.bronze.vra
"""))

linhas,partida_real_null_de_verdade,partida_real_string_null,partida_prevista_string_null,com_fracao_de_segundo
1597255,0,45895,45136,114283


In [0]:
# CTE permite transformações mais agressivas. É melhor em performace e legibilidade

display (spark.sql("""
    CREATE OR REPLACE TABLE voebem.silver.vra AS
    WITH tipado AS(
        SELECT
            icao_empresa_aerea,
            numero_voo,
            codigo_autorizacao_di,
            codigo_tipo_linha,
            icao_aerodromo_origem,
            icao_aerodromo_destino,
            try_cast(nullif(partida_prevista, 'null') AS TIMESTAMP) AS partida_prevista,
            try_cast(nullif(partida_real, 'null') AS TIMESTAMP) AS partida_real,
            try_cast(nullif(chegada_prevista, 'null') AS TIMESTAMP) AS chegada_prevista,
            try_cast(nullif(chegada_real, 'null') AS TIMESTAMP) AS chegada_real,
            situacao_voo,
            nullif(codigo_justificativa, 'N/A') AS codigo_justificativa,
            _arquivo_origem,
            _ingerido_em
            FROM voebem.bronze.vra
    )
    SELECT 
    icao_empresa_aerea,
    numero_voo,
    codigo_autorizacao_di,
    codigo_tipo_linha,
    icao_aerodromo_origem,
    icao_aerodromo_destino,

    partida_prevista,
    CAST(partida_prevista AS DATE) AS partida_prevista_data,
    date_format(partida_prevista, 'HH:mm') AS partida_prevista_hora,

    partida_real,
    CAST(partida_real AS DATE) AS partida_real_data,
    date_format(partida_real, 'HH:mm' ) AS partida_real_hora,

    chegada_prevista,
    CAST(chegada_prevista AS DATE) AS chegada_prevista_data,
    date_format(chegada_prevista, 'HH:mm' ) AS chegada_prevista_hora,

    chegada_real,
    CAST(chegada_real AS DATE) AS chegada_real_data,
    date_format(chegada_real, 'HH:mm' ) AS chegada_real_hora,

    situacao_voo,
    codigo_justificativa,

    -- subtração de colunas da própria linha, sem limiar e sem decisão (para verificar atrasos ou adiantamentos de partida e chegada)
    CAST(timestampdiff(MINUTE, partida_prevista, partida_real) AS INT) AS atraso_partida_min,
    CAST(timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS atraso_chegada_min,
    CAST(timestampdiff(MINUTE, partida_prevista, partida_real) - timestampdiff(MINUTE, chegada_prevista, chegada_real) AS INT) AS minutos_recuperados,

    _arquivo_origem,
    _ingerido_em, current_timestamp() AS _transformado_em
    FROM tipado
"""))

print("Silver.vra criada")

num_affected_rows,num_inserted_rows


Silver.vra criada


In [0]:
# verificando se a tipagem deu certo. A diferença entre silver e bronze deve ser 0 linhas

display(spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM voebem.bronze.vra) AS bronze_vra,
        (SELECT COUNT(*) FROM voebem.silver.vra) AS silver_vra,
        (SELECT COUNT(*) FROM voebem.bronze.vra) - (SELECT COUNT(*) FROM voebem.silver.vra) AS diferenca
"""))

bronze_vra,silver_vra,diferenca
1597255,1597255,0


In [0]:
display(spark.sql("""
    SELECT
        COUNT(partida_prevista) AS partida_prevista_ok,
        COUNT(partida_real) AS partida_real_ok,
        COUNT(chegada_prevista) AS chegada_prevista_ok,
        COUNT(chegada_real) AS chegada_real_ok,
        COUNT(minutos_recuperados) AS minutos_recuperados_ok
    FROM voebem.silver.vra
"""))


partida_prevista_ok,partida_real_ok,chegada_prevista_ok,chegada_real_ok,minutos_recuperados_ok
1552119,1551360,1552119,1551360,1506224


In [0]:
display(spark.sql("""
    SELECT icao_empresa_aerea, numero_voo, icao_aerodromo_origem, icao_aerodromo_destino, partida_prevista, partida_prevista_data, partida_prevista_hora, atraso_partida_min, atraso_chegada_min, minutos_recuperados, situacao_voo
    FROM voebem.silver.vra
    ORDER BY partida_prevista
"""))

icao_empresa_aerea,numero_voo,icao_aerodromo_origem,icao_aerodromo_destino,partida_prevista,partida_prevista_data,partida_prevista_hora,atraso_partida_min,atraso_chegada_min,minutos_recuperados,situacao_voo
GLO,Z7614,SBGR,SACO,null,null,null,null,null,null,REALIZADO
GLO,Z7752,SBRF,SBRF,null,null,null,null,null,null,REALIZADO
GLO,Z9029,SBSV,SBSV,null,null,null,null,null,null,REALIZADO
GLO,Z7749,KMIA,SBBR,null,null,null,null,null,null,REALIZADO
GLO,Z7632,SBFL,SAEZ,null,null,null,null,null,null,REALIZADO
GLO,Z7749,KMIA,SBBR,null,null,null,null,null,null,REALIZADO
GLO,Z7692,SAEZ,SABE,null,null,null,null,null,null,REALIZADO
GLO,Z9026,SBBR,SBMO,null,null,null,null,null,null,REALIZADO
GLO,Z7612,SBGL,SACO,null,null,null,null,null,null,REALIZADO
GLO,Z7650,SBGR,SABE,null,null,null,null,null,null,REALIZADO


In [0]:
# Juntando duas tabelas com dados iguais (uma para nacional e outra para estrangeira), mas no momento de analisar, o profissional provavelmente ira querer ver ambas de uma vez, e não ficar pesquisando uma por vez

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.empresas AS
SELECT
   icao,
   sigla_iata,
   razao_social,
   servico,
   cidade,
   uf, 
   situacao,
   'nacional' AS origem_cadastro,
   _arquivo_origem,
   _ingerido_em,
   current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_aereas_nacionais

UNION ALL

SELECT
   icao,
   sigla_iata,
   razao_social,
   servico,
   cidade,
   uf, 
   situacao,
   'estrangeira' AS origem_cadastro,
   _arquivo_origem,
   _ingerido_em,
   current_timestamp() AS _transformado_em
FROM voebem.bronze.empresas_aereas_estrangeiras
""")

# Apenas verificando se não modifiquei a quantidade de linhas das tabelas originais
display(spark.sql("""
    SELECT 
        (SELECT COUNT(*) FROM voebem.bronze.empresas_aereas_nacionais) AS bronze_nacionais,
        (SELECT COUNT(*) FROM voebem.bronze.empresas_aereas_estrangeiras) AS bronze_estrangeiras,
        (SELECT COUNT(*) FROM voebem.bronze.empresas_aereas_nacionais) + (SELECT COUNT(*) FROM voebem.bronze.empresas_aereas_estrangeiras) AS soma_esperada,
        (SELECT COUNT(*) FROM voebem.silver.empresas) AS silver_empresas
"""))

bronze_nacionais,bronze_estrangeiras,soma_esperada,silver_empresas
729,150,879,879


In [0]:
display(spark.sql("""
    SELECT origem_cadastro,
        COUNT(*) AS linhas,
        COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM voebem.silver.empresas
    GROUP BY origem_cadastro
    ORDER BY origem_cadastro
"""))

origem_cadastro,linhas,com_icao
estrangeira,150,149
nacional,729,20


In [0]:
#codigo_oaci em bronze.aerodromos corresponde a icao  nos demais

spark.sql("""
CREATE OR REPLACE TABLE voebem.silver.aerodromos AS
SELECT
    codigo_oaci,
    ciad,
    nome,
    municipio,
    uf AS uf_nome,
    municipio_servido,
    uf_servido  AS uf_servido_nome,
    latitude AS latitude_dms,
    longitude AS longitude_dms,
    try_cast(replace(altitude, ',' , '.') AS DOUBLE) AS altitude_m,
    operacao_diurna,
    operacao_noturna,
    situacao,
    _ingerido_em,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.aerodromos
""")

spark.sql("""
    CREATE OR REPLACE TABLE voebem.silver.codigos_operacao AS SELECT
    dominio,
    codigo,
    descricao,
    current_timestamp() AS _transformado_em
FROM voebem.bronze.codigos_operacao 
""")

display(spark.sql("""
    SELECT 'aerodromos' AS tabela,
        (SELECT COUNT(*) FROM voebem.bronze.aerodromos) AS bronze,
        (SELECT COUNT(*) FROM voebem.silver.aerodromos) AS silver
    UNION ALL
    SELECT 'codigos_operacao',
        (SELECT COUNT(*) FROM voebem.bronze.codigos_operacao),
        (SELECT COUNT(*) FROM voebem.silver.codigos_operacao)
"""))

tabela,bronze,silver
aerodromos,496,496
codigos_operacao,13,13


In [0]:
COMENTARIOS_VRA = {
    "icao_empresa_aerea":          "Codigo ICAO de tres letras da empresa aerea que operou a etapa. Chave para silver.empresas.",
    "numero_voo":            "Numero do voo divulgado pela companhia. Identificador comercial, nao numerico: pode ter zero a esquerda e se repete entre datas.",
    "codigo_autorizacao_di":             "Codigo de autorizacao (DI) da etapa: distingue etapa regular, extra, de retorno, charter. Descricao em silver.codigos_operacao (dominio codigo_di).",
    "codigo_tipo_linha":     "Codigo do tipo de linha: N e C domesticas, I e G internacionais. Descricao em silver.codigos_operacao (dominio codigo_tipo_linha).",
    "icao_aerodromo_origem":           "Codigo ICAO do aerodromo de onde a etapa partiu. Chave para silver.aerodromos - aeroportos estrangeiros nao constam no cadastro da ANAC.",
    "icao_aerodromo_destino":          "Codigo ICAO do aerodromo onde a etapa pousou. Mesma observacao de cobertura da origem.",
    "partida_prevista":      "Horario de partida programado pela companhia, na hora local do aeroporto de origem.",
    "partida_prevista_data": "Data da partida programada, separada para facilitar analise por dia.",
    "partida_prevista_hora": "Hora e minuto da partida programada (HH:mm), separada para analise por faixa horaria.",
    "partida_real":          "Horario em que a aeronave efetivamente saiu. Nulo em voo cancelado, que nao chegou a partir.",
    "partida_real_data":     "Data da partida efetiva.",
    "partida_real_hora":     "Hora e minuto da partida efetiva (HH:mm).",
    "chegada_prevista":      "Horario de chegada programado, na hora local do aeroporto de destino.",
    "chegada_prevista_data": "Data da chegada programada.",
    "chegada_prevista_hora": "Hora e minuto da chegada programada (HH:mm).",
    "chegada_real":          "Horario em que a aeronave efetivamente pousou. Nulo em voo cancelado.",
    "chegada_real_data":     "Data da chegada efetiva.",
    "chegada_real_hora":     "Hora e minuto da chegada efetiva (HH:mm).",
    "situacao_voo":          "Situacao informada pela companhia: REALIZADO quando a etapa aconteceu, CANCELADO quando nao.",
    "codigo_justificativa":  "Motivo declarado do atraso. Deixou de ser exigido pela ANAC em abril de 2020 com a revogacao da IAC 1504: vem vazio em toda a janela deste projeto.",
    "atraso_partida_min":    "Minutos entre a partida programada e a partida efetiva. Positivo e atraso, negativo e antecipacao. Aritmetica pura: nao aplica limiar de pontualidade.",
    "atraso_chegada_min":    "Minutos entre a chegada programada e a chegada efetiva. Positivo e atraso, negativo e antecipacao.",
    "minutos_recuperados":   "Minutos que a etapa recuperou em voo: atraso de partida menos atraso de chegada. Positivo significa que chegou menos atrasada do que saiu.",
    "_arquivo_origem":       "Auditoria: nome do arquivo CSV mensal da ANAC de onde a linha veio.",
    "_ingerido_em":          "Auditoria: momento em que a linha entrou no bronze.",
    "_transformado_em":      "Auditoria: momento em que a silver foi reconstruida a partir do bronze.",
}

for coluna, comentario in COMENTARIOS_VRA.items():
    spark.sql(f" ALTER TABLE voebem.silver.vra ALTER COLUMN {coluna} COMMENT '{comentario}' ")

print(f"{len(COMENTARIOS_VRA)}   colunas comentadas em silver.vra")

26   colunas comentadas em silver.vra


In [0]:
COMENTARIOS_EMPRESAS = {
    "icao":              "Codigo ICAO de tres letras da empresa. Vazio para operadores sem codigo (aviacao agricola, taxi aereo, aeroclube).",
    "sigla_iata":        "Sigla de duas letras da empresa no padrao IATA, como publicada pela ANAC.",
    "razao_social":      "Razao social da empresa aerea. E o nome que aparece para quem consome o produto final.",
    "servico":           "Tipo de servico autorizado pela ANAC: transporte regular, nao regular, aeroagricola, taxi aereo.",
    "cidade":            "Municipio da sede ou do representante legal no Brasil.",
    "uf":                "Sigla da unidade federativa da sede.",
    "situacao":          "Situacao do registro na ANAC: ATIVA ou nao. Registro inativo permanece na tabela porque a empresa pode ter voado no periodo analisado.",
    "origem_cadastro":   "De qual dos dois cadastros da ANAC este registro veio: nacional ou estrangeira. E a coluna que preserva a fronteira entre as duas fontes depois da uniao.",
    "_arquivo_origem":   "Auditoria: arquivo CSV de origem.",
    "_ingerido_em":      "Auditoria: momento da ingestao no bronze.",
    "_transformado_em":  "Auditoria: momento da construcao da silver.",
}

COMENTARIOS_AERODROMOS = {
    "codigo_oaci":       "Codigo ICAO (OACI) do aerodromo. Chave de ligacao com origem e destino do VRA.",
    "ciad":              "Codigo de identificacao do aerodromo no cadastro da ANAC.",
    "nome":              "Nome do aerodromo como publicado pela ANAC.",
    "municipio":         "Municipio onde o aerodromo esta fisicamente localizado.",
    "uf_nome":           "Nome da unidade federativa POR EXTENSO (Acre, Sao Paulo), nao a sigla: e assim que a ANAC publica.",
    "municipio_servido": "Municipio principal atendido pelo aerodromo, que pode ser diferente do municipio onde ele fica.",
    "uf_servido_nome":   "Nome por extenso da UF do municipio servido.",
    "latitude_dms":      "Latitude em graus, minutos e segundos, como publicada pela ANAC.",
    "longitude_dms":     "Longitude em graus, minutos e segundos, como publicada pela ANAC.",
    "altitude_m":        "Altitude do aerodromo em metros. Na origem vem com virgula decimal.",
    "operacao_diurna":   "Indica se o aerodromo esta autorizado a operar em horario diurno.",
    "operacao_noturna":  "Indica se o aerodromo esta autorizado a operar em horario noturno.",
    "situacao":          "Situacao do aerodromo no cadastro da ANAC.",
    "_ingerido_em":      "Auditoria: momento da ingestao no bronze.",
    "_transformado_em":  "Auditoria: momento da construcao da silver.",
}

COMENTARIOS_CODIGOS = {
    "dominio":           "A qual coluna do VRA este codigo pertence: codigo_di ou codigo_tipo_linha.",
    "codigo":            "O codigo como aparece no VRA.",
    "descricao":         "Descricao oficial do codigo, curada da pagina de descricao de variaveis da ANAC.",
    "_transformado_em":  "Auditoria: momento da construcao da silver.",
}

for tabela, mapa in [
    ("voebem.silver.empresas",         COMENTARIOS_EMPRESAS),
    ("voebem.silver.aerodromos",       COMENTARIOS_AERODROMOS),
    ("voebem.silver.codigos_operacao", COMENTARIOS_CODIGOS),
]:
    for coluna, comentario in mapa.items():
        spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    print(f"{len(mapa)} colunas comentadas em {tabela}")

11 colunas comentadas em voebem.silver.empresas
15 colunas comentadas em voebem.silver.aerodromos
4 colunas comentadas em voebem.silver.codigos_operacao


In [0]:
TABELAS = {
    "voebem.silver.vra": (
        "Silver - espelho governado de bronze.vra. Mesmo grao (uma linha por etapa de voo) e "
        "MESMA contagem de linhas do bronze: sem filtro, sem agregacao e sem regra de negocio. "
        "Traz tipagem, data e hora separadas e as tres metricas de aritmetica pura de atraso. "
        "Pontualidade, escopo e exclusoes ficam na gold.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-VRA", "grao": "etapa_de_voo"},
    ),
    "voebem.silver.empresas": (
        "Silver - cadastro unificado de empresas aereas: uniao dos dois cadastros do bronze "
        "(nacionais e estrangeiras) com a coluna origem_cadastro preservando a fonte de cada registro. "
        "Contagem igual a soma exata das duas tabelas de origem.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-Operador-Aereo", "grao": "empresa"},
    ),
    "voebem.silver.aerodromos": (
        "Silver - espelho governado do cadastro de aerodromos publicos da ANAC. Cobre apenas "
        "aerodromos brasileiros: aeroportos estrangeiros do VRA nao constam aqui, e isso e "
        "propriedade da fonte, nao defeito.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-Aerodromos", "grao": "aerodromo"},
    ),
    "voebem.silver.codigos_operacao": (
        "Silver - espelho da seed table de codigos de operacao (DI e tipo de linha) com as "
        "descricoes oficiais da ANAC.",
        {"camada": "silver", "dominio": "aviacao", "fonte": "ANAC-seed", "grao": "codigo"},
    ),
}

for tabela, (comentario, tags) in TABELAS.items():
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    pares = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    spark.sql(f"ALTER TABLE {tabela} SET TAGS ({pares})")
    print(f"{tabela}: comentario + {len(tags)} tags")

voebem.silver.vra: comentario + 4 tags
voebem.silver.empresas: comentario + 4 tags
voebem.silver.aerodromos: comentario + 4 tags
voebem.silver.codigos_operacao: comentario + 4 tags


In [0]:
display(spark.sql("""
    SELECT table_name,
        COUNT(*)                                                                          AS colunas,
        SUM(CASE WHEN comment IS NULL OR comment = '' THEN 1 ELSE 0 END)                  AS sem_comentario,
        ROUND(100.0 * SUM(CASE WHEN comment IS NOT NULL AND comment <> '' THEN 1 ELSE 0 END)
            / COUNT(*), 1)                                                                AS pct_documentado
    FROM voebem.information_schema.columns
    WHERE table_schema = 'silver'
    GROUP BY table_name
    ORDER BY table_name
"""))

table_name,colunas,sem_comentario,pct_documentado
aerodromos,15,0,100.0
codigos_operacao,4,0,100.0
empresas,11,0,100.0
vra,26,0,100.0


In [0]:
display(spark.sql("""
    SELECT table_name, tag_name, tag_value
    FROM voebem.information_schema.table_tags
    WHERE schema_name = 'silver'
    ORDER BY table_name, tag_name        
"""))

table_name,tag_name,tag_value
aerodromos,camada,silver
aerodromos,dominio,aviacao
aerodromos,fonte,ANAC-Aerodromos
aerodromos,grao,aerodromo
codigos_operacao,camada,silver
codigos_operacao,dominio,aviacao
codigos_operacao,fonte,ANAC-seed
codigos_operacao,grao,codigo
empresas,camada,silver
empresas,dominio,aviacao


In [0]:
display(spark.sql("SHOW TABLES IN voebem.silver"))

database,tableName,isTemporary
silver,aerodromos,false
silver,codigos_operacao,false
silver,empresas,false
silver,vra,false


In [0]:
# QA 1: Inconsistencias entre situacao_voo e timestamps
# Voos REALIZADO deveriam ter partida_real e chegada_real; CANCELADO nao.

display(spark.sql("""
    SELECT
        situacao_voo,
        COUNT(*) AS total,
        SUM(CASE WHEN partida_prevista IS NULL THEN 1 ELSE 0 END) AS sem_partida_prevista,
        SUM(CASE WHEN partida_real IS NULL THEN 1 ELSE 0 END) AS sem_partida_real,
        SUM(CASE WHEN chegada_prevista IS NULL THEN 1 ELSE 0 END) AS sem_chegada_prevista,
        SUM(CASE WHEN chegada_real IS NULL THEN 1 ELSE 0 END) AS sem_chegada_real,
        SUM(CASE WHEN partida_prevista IS NULL AND partida_real IS NULL AND chegada_prevista IS NULL AND chegada_real IS NULL THEN 1 ELSE 0 END) AS todos_timestamps_nulos
    FROM voebem.silver.vra
    GROUP BY situacao_voo
    ORDER BY total DESC
"""))

situacao_voo,total,sem_partida_prevista,sem_partida_real,sem_chegada_prevista,sem_chegada_real,todos_timestamps_nulos
REALIZADO,1551360,45136,0,45136,0,0
CANCELADO,45895,0,45895,0,45895,0


In [0]:
# QA 2: Integridade referencial — chaves que nao casam com dimensoes

display(spark.sql("""
    WITH vra AS (SELECT icao_empresa_aerea, icao_aerodromo_origem, icao_aerodromo_destino FROM voebem.silver.vra)
    SELECT
        'empresa_aerea -> silver.empresas' AS verificacao,
        COUNT(*) AS total_linhas_vra,
        SUM(CASE WHEN e.icao IS NULL THEN 1 ELSE 0 END) AS sem_match,
        ROUND(100.0 * SUM(CASE WHEN e.icao IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_cobertura
    FROM vra v
    LEFT JOIN voebem.silver.empresas e ON v.icao_empresa_aerea = e.icao

    UNION ALL

    SELECT
        'aerodromo_origem -> silver.aerodromos' AS verificacao,
        COUNT(*),
        SUM(CASE WHEN a.codigo_oaci IS NULL THEN 1 ELSE 0 END),
        ROUND(100.0 * SUM(CASE WHEN a.codigo_oaci IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2)
    FROM vra v
    LEFT JOIN voebem.silver.aerodromos a ON v.icao_aerodromo_origem = a.codigo_oaci

    UNION ALL

    SELECT
        'aerodromo_destino -> silver.aerodromos',
        COUNT(*),
        SUM(CASE WHEN a.codigo_oaci IS NULL THEN 1 ELSE 0 END),
        ROUND(100.0 * SUM(CASE WHEN a.codigo_oaci IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2)
    FROM vra v
    LEFT JOIN voebem.silver.aerodromos a ON v.icao_aerodromo_destino = a.codigo_oaci
"""))

verificacao,total_linhas_vra,sem_match,pct_cobertura
empresa_aerea -> silver.empresas,1597255,3034,99.81
aerodromo_origem -> silver.aerodromos,1597255,162951,89.80
aerodromo_destino -> silver.aerodromos,1597255,163526,89.76


In [0]:
# QA 3: Duplicidades — mesma empresa, voo, data de partida prevista, origem e destino

display(spark.sql("""
    WITH duplicatas AS (
        SELECT
            icao_empresa_aerea,
            numero_voo,
            partida_prevista_data,
            icao_aerodromo_origem,
            icao_aerodromo_destino,
            COUNT(*) AS repeticoes
        FROM voebem.silver.vra
        WHERE partida_prevista_data IS NOT NULL
        GROUP BY icao_empresa_aerea, numero_voo, partida_prevista_data, icao_aerodromo_origem, icao_aerodromo_destino
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*) AS grupos_duplicados,
        SUM(repeticoes) AS total_linhas_envolvidas,
        MAX(repeticoes) AS max_repeticoes,
        ROUND(100.0 * SUM(repeticoes) / (SELECT COUNT(*) FROM voebem.silver.vra), 2) AS pct_duplicadas
    FROM duplicatas
"""))

grupos_duplicados,total_linhas_envolvidas,max_repeticoes,pct_duplicadas
225,483,4,0.03


In [0]:
# QA 4: Atrasos extremos e incoerencias temporais

display(spark.sql("""
    SELECT
        SUM(CASE WHEN atraso_partida_min > 360 THEN 1 ELSE 0 END) AS atraso_partida_gt_6h,
        SUM(CASE WHEN atraso_partida_min < -60 THEN 1 ELSE 0 END) AS partida_adiantada_gt_1h,
        SUM(CASE WHEN atraso_chegada_min > 360 THEN 1 ELSE 0 END) AS atraso_chegada_gt_6h,
        SUM(CASE WHEN atraso_chegada_min < -60 THEN 1 ELSE 0 END) AS chegada_adiantada_gt_1h,
        SUM(CASE WHEN partida_real IS NOT NULL AND chegada_real IS NOT NULL
                  AND partida_real > chegada_real THEN 1 ELSE 0 END) AS partida_depois_chegada,
        SUM(CASE WHEN ABS(minutos_recuperados) > 240 THEN 1 ELSE 0 END) AS recuperacao_gt_4h,
        MIN(atraso_partida_min) AS min_atraso_partida,
        MAX(atraso_partida_min) AS max_atraso_partida,
        MIN(atraso_chegada_min) AS min_atraso_chegada,
        MAX(atraso_chegada_min) AS max_atraso_chegada
    FROM voebem.silver.vra
"""))

atraso_partida_gt_6h,partida_adiantada_gt_1h,atraso_chegada_gt_6h,chegada_adiantada_gt_1h,partida_depois_chegada,recuperacao_gt_4h,min_atraso_partida,max_atraso_partida,min_atraso_chegada,max_atraso_chegada
3862,1836,3809,3094,0,141,-43057,44855,-43082,44854


In [0]:
# QA 5: Qualidade dos cadastros

# 5a: Empresas — nulidade de ICAO por origem_cadastro e situacao
display(spark.sql("""
    SELECT
        origem_cadastro,
        situacao,
        COUNT(*) AS total,
        SUM(CASE WHEN icao IS NULL OR icao = '' THEN 1 ELSE 0 END) AS sem_icao,
        SUM(CASE WHEN sigla_iata IS NULL OR sigla_iata = '' THEN 1 ELSE 0 END) AS sem_iata
    FROM voebem.silver.empresas
    GROUP BY origem_cadastro, situacao
    ORDER BY origem_cadastro, situacao
"""))

# 5b: Aerodromos — nulidade de coordenadas e situacao
display(spark.sql("""
    SELECT
        situacao,
        COUNT(*) AS total,
        SUM(CASE WHEN latitude_dms IS NULL OR latitude_dms = '' THEN 1 ELSE 0 END) AS sem_latitude,
        SUM(CASE WHEN longitude_dms IS NULL OR longitude_dms = '' THEN 1 ELSE 0 END) AS sem_longitude,
        SUM(CASE WHEN altitude_m IS NULL THEN 1 ELSE 0 END) AS sem_altitude
    FROM voebem.silver.aerodromos
    GROUP BY situacao
    ORDER BY total DESC
"""))

origem_cadastro,situacao,total,sem_icao,sem_iata
estrangeira,ATIVA,150,1,1
nacional,ATIVA,729,709,151


situacao,total,sem_latitude,sem_longitude,sem_altitude
Cadastrado,465,0,0,0
Interditado,31,0,0,0


# Diagnostico de Qualidade da Camada Silver

## Resumo dos achados

| # | Categoria | Achado | Impacto | Severidade |
|---|-----------|--------|---------|------------|
| 1 | Nulos VRA | 45.136 voos REALIZADO sem `partida_prevista` e `chegada_prevista` | Nao da para calcular atraso dessas etapas (~3% do total) | Medio |
| 2 | Nulos VRA | 45.895 voos CANCELADO sem `partida_real`/`chegada_real` | Esperado: voo cancelado nao partiu. Nao e defeito. | Nenhum |
| 3 | Integridade FK | 3.034 linhas do VRA sem match em `silver.empresas` (99,81% de cobertura) | Algumas empresas operaram mas nao constam no cadastro | Baixo |
| 4 | Integridade FK | ~163 mil linhas sem match em `silver.aerodromos` (10% do VRA) | Esperado: aeroportos estrangeiros nao estao no cadastro da ANAC | Nenhum |
| 5 | Duplicidades | 225 grupos duplicados (483 linhas, 0,03% do total) | Pode inflar contagens na gold se nao tratado | Medio |
| 6 | Outliers atraso | Atrasos de partida/chechada variando de -43h a +45h (~31 dias) | Valores absurdos provavelmente erro de digitacao ou reescalonamento cross-day | Alto |
| 7 | Outliers atraso | 3.862 atrasos de partida > 6h; 3.809 atrasos de chegada > 6h | Pode ser legtimo (atraso extremo) ou erro | Medio |
| 8 | Coerencia temporal | 0 casos de `partida_real > chegada_real` | Coerencia temporal preservada | Nenhum |
| 9 | Empresas | 709 de 729 empresas nacionais sem codigo ICAO | Esperado: aviacao agricola/taxi aereo nao tem ICAO. So 20 operadores regulares tem | Nenhum |
| 10 | Aerodromos | 31 aerodromos Interditado; 0 sem coordenadas ou altitude | Cadastro limpo e completo | Nenhum |

## Tratamentos recomendados para a Gold

### 1. Nulidade de partida/chegada prevista em voos REALIZADO
**Causa**: a companhia nao informou o horario previsto para essas etapas.
**Tratamento na gold**: criar flag `tem_prevista` (boolean) para documentar a ausencia. Nao tentar imputar — nao ha base para inferir o horario programado.

### 2. Duplicidades
**Causa**: provavel reenvio de arquivo mensal pela ANAC ou reprocessamento de bronze.
**Tratamento na gold**: deduplicar por `(icao_empresa_aerea, numero_voo, partida_prevista_data, icao_aerodromo_origem, icao_aerodromo_destino)` mantendo a linha mais recente (`_ingerido_em` maximo). Adicionar verificacao no pipeline de bronze para impedir reingestao do mesmo arquivo.

### 3. Outliers de atraso (±31 dias)
**Causa**: provavel erro de data na origem (digitacao DD/MM trocado) ou reescalonamento onde o voo passou para o dia seguinte/mes seguinte.
**Tratamento na gold**: criar flag `atraso_suspeito` quando `ABS(atraso_partida_min) > 720` (12 horas) ou `ABS(atraso_chegada_min) > 720`. Investigar amostra antes de descartar. Considerar truncar para NULL se confirmado erro.

### 4. Integridade referencial de empresa
**Causa**: 3.034 linhas do VRA referenciam ICAO que nao existe em `silver.empresas`.
**Tratamento na gold**: manter LEFT JOIN (nao INNER) para nao perder voos. Documentar que ~0,19% das etapas ficam sem nome da empresa. Considerar enriquecer o cadastro com os ICAOs faltantes.

### 5. Integridade referencial de aerodromo (estrangeiros)
**Causa**: aeroportos estrangeiros nao estao no cadastro da ANAC — e propriedade da fonte.
**Tratamento na gold**: nada a fazer na silver. Na gold, criar dim de aerodromos enriquecida (ANAC + fonte externa de ICAO estrangeiros) se a analise exigir nomes de aeroportos internacionais.

### 6. Empresas nacionais sem ICAO
**Causa**: cadastro da ANAC inclui operadores nao regulares (agricola, taxi aereo, aeroclube) que nao tem codigo ICAO.
**Tratamento na gold**: nenhum. O join com VRA ja filtra naturalmente — so operadores com ICAO aparecem no VRA.

## O que NAO precisa tratamento

* Voos CANCELADO sem timestamps reais — e o comportamento esperado.
* Aerodromos interditados no cadastro — continuam valido para analise historica.
* Coerencia temporal partida <= chegada — ja esta preservada (0 violacoes).
* Documentacao e tags — 100% das colunas e tabelas documentadas.